# Credit Risk | Aggregate Forecasting (Optional)

This is a **separate exercise** from individual-loan probability of default (PD). Here we forecast one monthly default-rate time series. Keep the EDA and PD-classification work in `01_credit_risk_eda_simple_en.ipynb`.

## 1. Load monthly data from PostgreSQL

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Project paths: notebooks work when Jupyter starts from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")
required = ["DB_USER", "DB_PASSWORD", "DB_HOST", "DB_NAME"]
if all(os.getenv(key) for key in required):
    db_url = URL.create(
        "postgresql+psycopg", username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"), host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT", "5432")), database=os.getenv("DB_NAME"),
    )
    performance = pd.read_sql("SELECT * FROM raw.monthly_performance", create_engine(db_url))
    print("Source: PostgreSQL")
else:
    performance = pd.read_csv(PROJECT_ROOT / "data/raw/desempenho_mensal.csv")
    performance = performance.rename(columns={
        "mes_referencia": "reference_month", "novo_default": "new_default"
    })
    print("Source: bundled synthetic CSV")

performance["reference_month"] = pd.to_datetime(performance["reference_month"])
monthly = performance.groupby("reference_month").agg(
    observed_loans=("loan_id", "count"),
    new_defaults=("new_default", "sum")
)
series = (100 * monthly.new_defaults / monthly.observed_loans).sort_index().asfreq("MS")
assert not series.isna().any(), "Missing monthly periods found"

series.plot(figsize=(10, 4), title="Monthly new-default rate")
plt.ylabel("Rate (%)")
plt.show()


## 2. Hold out six months and fit Holt's model

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error

train = series.iloc[:-6]
test = series.iloc[-6:]

holt = ExponentialSmoothing(train, trend="add", damped_trend=True).fit()
pred = holt.forecast(len(test)).clip(lower=0, upper=100)
baseline = pd.Series(train.iloc[-1], index=test.index)

print(f"Holt MAE: {mean_absolute_error(test, pred):.4f} percentage points")
print(f"Baseline MAE: {mean_absolute_error(test, baseline):.4f} percentage points")

ax = train.plot(figsize=(10, 4), label="Train")
test.plot(ax=ax, label="Actual")
pred.plot(ax=ax, label="Holt forecast", style="--o")
ax.set_ylabel("Rate (%)")
ax.legend()
plt.show()

## 3. Forecast the next three months

Retrain on the full history, then forecast. Treat these as illustrative predictions from synthetic data.

In [ ]:
holt = ExponentialSmoothing(series, trend="add", damped_trend=True).fit()
forecast = holt.forecast(3).clip(lower=0, upper=100)
display(forecast.to_frame("predicted_rate_pct").round(3))

ax = series.plot(figsize=(10, 4), label="History")
forecast.plot(ax=ax, label="Forecast", style="--o")
ax.set_title("Three-month aggregate default-rate forecast")
ax.set_ylabel("Rate (%)")
ax.legend()
plt.show()